<a href="https://colab.research.google.com/github/MartVASS/MaskArchitectureAnomaly_CourseProject/blob/main/notebooks/visualization_panoptic_cityscapes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Visualization panoptic segmentation on image from cityscapes


## Import github repo

In [ ]:
!git clone https://github.com/MartVASS/MaskArchitectureAnomaly_CourseProject.git
%cd MaskArchitectureAnomaly_CourseProject/eomt

In [ ]:
!pip install -r requirements.txt

## Import Cityscapes datatset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Add repo to the PYTHONPATH

In [ ]:
import sys
import os

sys.path.append(os.getcwd())

### Check if dataset is available


In [ ]:
data_path = "/content/drive/MyDrive/Cityscapes"

print(os.listdir(data_path))

## Preparation code

In [ ]:
import yaml
from lightning import seed_everything
import torch
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
import numpy as np
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import RepositoryNotFoundError
import warnings
import importlib

seed_everything(0, verbose=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # TODO: change to the GPU you want to use
print(f"device used :{device}")
img_idx = 15  # TODO: change to the index of the image you want to visualize
config_path = "configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"  # TODO: change to the config file
data_path = "/content/drive/MyDrive/Cityscapes"  # TODO: change to the dataset directory

with open(config_path, "r") as f:
    config = yaml.safe_load(f)


def create_mapping(images, ignore_index):
    unique_ids = np.unique(np.concatenate([np.unique(img) for img in images]))
    valid_ids = unique_ids[unique_ids != ignore_index]
    colors = np.array(
        [plt.cm.hsv(i / len(valid_ids))[:3] for i in range(len(valid_ids))]
    )
    mapping = {cid: colors[i] for i, cid in enumerate(valid_ids)}
    mapping[ignore_index] = np.array([0, 0, 0])
    return mapping


def apply_colormap(image, mapping):
    colored_image = np.zeros((*image.shape, 3))
    for cid in np.unique(image):
        colored_image[image == cid] = mapping.get(cid, [0, 0, 0])
    return colored_image

print("Ok STEP 1")

## Loading validation dataset

In [ ]:
print("Starting step 2 : Loading dataset...")

!unzip -q "/content/drive/MyDrive/Cityscapes/leftImg8bit_trainvaltest.zip" -d /content/

from PIL import Image
from torchvision import transforms

transform = transforms.ToTensor()

img_path = "/content/leftImg8bit/val/frankfurt/frankfurt_000000_000294_leftImg8bit.png"

img = Image.open(img_path).convert("RGB")
img = transform(img)

print(img.shape)

img = img.to(device)
img = img.unsqueeze(0)  # batch dimension

print("OK Step 2")


## Loading model

In [ ]:
print("Starting step 3 : Loading models...")

COCO_NUM_CLASSES = 133  # ou valeur du config COCO

# The original config does not directly contain 'img_size' under 'model.init_args',
# but it is likely inferred from the config file name 'eomt_base_640_2x.yaml'.
# Explicitly setting img_size based on this assumption.
img_size = 640

encoder_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)

encoder = encoder_cls(
    img_size=img_size, # Use the determined img_size
    **encoder_cfg.get("init_args", {})
)

network_cfg = config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)

network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}

network = network_cls(
    masked_attn_enabled=False,
    num_classes=COCO_NUM_CLASSES,
    encoder=encoder,
    **network_kwargs,
)

lit_module_name, lit_class_name = config["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)

model_kwargs = {k: v for k, v in config["model"]["init_args"].items() if k != "network"}
# Ensure img_size is passed to lit_cls as a tuple
model_kwargs['img_size'] = (img_size, img_size)
# Add 'stuff_classes' to model_kwargs for the MaskClassificationPanoptic model
model_kwargs['stuff_classes'] = config['data']['init_args']['stuff_classes']

model = (
    lit_cls(
        # Remove explicit img_size argument here, as it's already in model_kwargs
        num_classes=COCO_NUM_CLASSES,
        network=network,
        **model_kwargs, # img_size and stuff_classes will be unpacked from here
    )
    .eval()
    .to(device)
)

print("Step 3 OK")

## Load pre-trained weights from hugging face

In [ ]:

print("Starting step 4 : Load pre-trained weights from Hugging Face Hub")

name = config.get("trainer", {}).get("logger", {}).get("init_args", {}).get("name")

print(name)

if name is None:
    warnings.warn("No logger name found in the config. Please specify a model name.")
else:
    try:
        state_dict_path = hf_hub_download(
            repo_id=f"S362484/{name}",
            filename="eomt_coco.bin",
        )

        is_dinov3 = "dinov3" in name

        if is_dinov3:
            model_kwargs["ckpt_path"] = state_dict_path
            model_kwargs["delta_weights"] = True

        model = (
            lit_cls(
                # Remove explicit img_size argument here, as it's already in model_kwargs
                num_classes=133,
                network=network,
                **model_kwargs,
            )
            .eval()
            .to(device)
        )

        if not is_dinov3:
            state_dict = torch.load(
            state_dict_path,
            map_location=torch.device(device),
            weights_only=False
            )
            model.load_state_dict(state_dict, strict=False)

    except RepositoryNotFoundError:
        warnings.warn(
            f"Pre-trained model not found for `{name}`. Please load your own checkpoint."
        )

print("Step 4 OK")

## Plot prediction

In [ ]:
print("Starting step 5: plot prediction...")

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.nn import functional as F
from torch.amp import autocast
from PIL import Image
from torchvision import transforms


transform = transforms.ToTensor()

img = Image.open(img_path).convert("RGB")
img = transform(img)

img = img.to(device)
img = img.unsqueeze(0)  # batch dimension


def infer_panoptic(img):
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        # Convert float tensor (0-1) to uint8 tensor (0-255) for PIL.Image.fromarray
        # which is used internally by model.resize_and_pad_imgs_instance_panoptic
        img_for_model = (img.squeeze(0) * 255).byte()
        imgs = [img_for_model]
        img_sizes = [img.shape[-2:]]

        transformed_imgs = model.resize_and_pad_imgs_instance_panoptic(imgs)

        mask_logits_per_layer, class_logits_per_layer = model(transformed_imgs)

        mask_logits = F.interpolate(
            mask_logits_per_layer[-1],
            model.img_size,
            mode="bilinear"
        )

        mask_logits = model.revert_resize_and_pad_logits_instance_panoptic(
            mask_logits,
            img_sizes
        )

        preds = model.to_per_pixel_preds_panoptic(
            mask_logits,
            class_logits_per_layer[-1],
            model.stuff_classes,
            model.mask_thresh,
            model.overlap_thresh,
        )[0].cpu()

    pred = preds.numpy()
    sem_pred, inst_pred = pred[..., 0], pred[..., 1]

    return sem_pred, inst_pred


def draw_black_border(sem, inst, mapping):
    h, w = sem.shape
    out = np.zeros((h, w, 3))

    for s in np.unique(sem):
        out[sem == s] = mapping.get(s, [0, 0, 0])

    combined = sem.astype(np.int64) * 100000 + inst.astype(np.int64)

    border = np.zeros((h, w), dtype=bool)
    border[1:, :] |= combined[1:, :] != combined[:-1, :]
    border[:-1, :] |= combined[1:, :] != combined[:-1, :]
    border[:, 1:] |= combined[:, 1:] != combined[:, :-1]
    border[:, :-1] |= combined[:, 1:] != combined[:, :-1]

    out[border] = 0
    return out


def plot_panoptic_results(img, sem_pred, inst_pred):
    all_ids = np.unique(sem_pred)

    mapping = {
        s: (np.random.rand(3) if s >= 0 else [0, 0, 0])
        for s in all_ids
    }

    vis_pred = draw_black_border(sem_pred, inst_pred, mapping)

    img_np = img.squeeze(0).cpu().numpy().transpose(1, 2, 0)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].imshow(img_np)
    axes[0].set_title("Input (Cityscapes)")

    axes[1].imshow(vis_pred)
    axes[1].set_title("COCO Panoptic Prediction")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()



sem_pred, inst_pred = infer_panoptic(img)
plot_panoptic_results(img, sem_pred, inst_pred)

print("Step 5 OK")